# L7b: Stress Testing and Dynamic Portfolio Rebalancing
In this lecture, we examine why an optimized portfolio can drift away from its intended risk profile and develop a stress-tested, closed-loop rebalancing workflow.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * __Diagnose allocation drift:__ Explain how relative price movements, parameter uncertainty, and changing dependence structures move a portfolio away from its target allocation.
> * __Stress test an optimized portfolio:__ Evaluate drawdown, tail loss, turnover, and net present value under parameter, correlation, and transaction-cost shocks.
> * __Design a guarded rebalancing loop:__ Combine observations, allocation rules, trigger conditions, and risk limits in a repeatable portfolio-management process.

Let's connect portfolio optimization to the decisions required after deployment.
___


## Examples
We use the following examples to connect allocation theory with forward performance and rebalancing decisions:

> [▶ Analyze allocation drift in a maximum-Sharpe portfolio](./CHEME-5660-L7b-Example-Portfolio-Drift-Fall-2026.ipynb). This example measures how market movements change portfolio weights and risk exposures between rebalancing dates.
>
> [▶ Stress-test a minimum-variance portfolio](./CHEME-5660-L7b-Example-StressTest-MinVar-Fall-2026.ipynb). This example evaluates an optimized allocation under return, correlation, and transaction-cost shocks and summarizes terminal wealth, drawdown, failure probability, and turnover.

The first example establishes the problem; the stress-testing workflow supplies evidence for when and how to rebalance.
___


## Concept Review: Risky and Risk-Free Assets
In the last lecture, we introduced the problem of portfolios with a mixture of risky and risk-free assets. We discussed the Capital Allocation Line (CAL), which represents the risk-growth trade-off of a portfolio that combines a risk-free asset with a portfolio of risky assets. The slope of the CAL is determined by the __Sharpe Ratio__ of the risky asset portfolio, which measures the excess return per unit of risk.

The minimum variance portfolio problem for a portfolio $\mathcal{P}$ using the single index model with a combination of risky and risk-free assets is given by:
$$
\begin{align*}
\text{minimize}~\operatorname{Var}(g_{\mathcal{P}}) &= \sum_{i\in\mathcal{P}}\sum_{j\in\mathcal{P}}w_{i}w_{j}
\operatorname{Cov}\left(g_{i},g_{j}\right) \\
\text{subject to}~\mathbb{E}[g_{\mathcal{P}}]& = w_{f}g_{f}+\alpha_{\mathcal{P}}+\beta_{\mathcal{P}}\;\mathbb{E}[g_{M}] = {g^{*}}\\
\alpha_{\mathcal{P}} & = \sum_{i\in\mathcal{P}}w_{i}\;\alpha_{i}\\
\beta_{\mathcal{P}} & = \sum_{i\in\mathcal{P}}w_{i}\;\beta_{i} \\
w_{f}+\sum_{i\in\mathcal{P}}w_{i} & = 1 \\
w_{f}&\geq{0}\\
w_{i}&\geq{0}\qquad{\forall{i}\in\mathcal{P}}
\end{align*}
$$
where the covariance between assets $i$ and $j$ is given by:
$$
\begin{align*}
\operatorname{Cov}(g_{i}, g_{j}) & = \begin{cases}
\beta_{i}^{2}\sigma_{m}^{2}+\Delta{t}\;\sigma_{\epsilon_{i}}^{2} & i = j \\
\beta_{i}\beta_{j}\sigma_{m}^2 & i \neq j
\end{cases} \\
\end{align*}
$$

The terms $w_{i}\geq{0}$ denote the fraction of risky asset $i\in\mathcal{P}$, 
the quantity $w_{f}$ denotes the fraction of risk-free assets in the portfolio, 
$g_{f}$ denotes the risk-free rate or return, and $g^{*}$ is the minimum required growth rate (return) 
for the overall portfolio $\mathcal{P}$. 

> __The Separation Theorem__
>
> Every investor, regardless of their risk preferences, should hold the same risky portfolio, i.e., the tangent portfolio. What differs between investors is how much they allocate between this tangent portfolio and the risk-free asset. Risk-averse investors hold more of the risk-free asset, while risk-seeking investors may borrow money (making $w_f$ negative) to invest more than 100% of their wealth in the tangent portfolio.
>
> This result is known as the __Separation Theorem__ or __Two-Fund Separation Theorem__, originally developed by James Tobin in his 1958 paper on liquidity preference. [Tobin, James (1958). "Liquidity Preference as Behavior Towards Risk". The Review of Economic Studies, 25(2): 65-86.](https://doi.org/10.2307/2296205) The investment decision (which risky portfolio to hold) is separated from the financing decision (how much to borrow or lend at the risk-free rate).


Let's finish up our example of portfolios with risky and risk-free assets.

> __Example__
>
> [▶ Let's build a portfolio with risky and risk-free assets](CHEME-5660-L6b-Example-SIM-MinVar-RRFA-Fall-2026.ipynb). In this example, we construct a portfolio that includes both risky assets and a risk-free asset, such as a treasury STRIPs bond. We will use the single index model to optimize the portfolio, aiming to minimize risk while achieving a specified return. We'll explore how the inclusion of a risk-free asset affects the portfolio's risk-growth profile, construct the Capital Allocation Line, and identify the tangent portfolio.

## CAL, the Tangent Portfolio, and the Sharpe Ratio

When we introduce a risk-free asset into our portfolio optimization problem, something remarkable happens: the efficient frontier transforms into a straight line in risk-growth space. This line is called the __Capital Allocation Line (CAL)__.

Under the model assumptions, the Capital Allocation Line describes portfolios formed by combining a horizon-matched risk-free payoff with return $g_f$ and zero variance over that horizon, and a tangent risky portfolio (denoted by $T$). A Treasury STRIP can approximate the risk-free payoff only when its maturity and the investor's horizon match and the position is held to maturity; before maturity its market value varies with rates. The tangent portfolio has expected growth rate $\mathbb{E}[g_T]$ and variance $\sigma_T^2$. Any portfolio on the CAL can be expressed as:
$$
\begin{align*}
\mathbb{E}[g_{\mathcal{P}}] &= w_f g_f + (1 - w_f)\;\mathbb{E}[g_T]\\
\sigma_{\mathcal{P}} &= (1 - w_f) \sigma_T
\end{align*}
$$
where $w_f$ is the fraction invested in the risk-free asset, $(1-w_f)$ is the fraction invested in the tangent portfolio, $\sigma_{\mathcal{P}} = \sqrt{\operatorname{Var}(g_{\mathcal{P}})}$ is the standard deviation of the portfolio growth rate, and $\sigma_T = \sqrt{\operatorname{Var}(g_T)}$ is the standard deviation of the tangent portfolio growth rate. To derive the CAL equation, we solve for $w_f$ from the second equation and substitute into the first:
$$
\begin{align*}
\sigma_{\mathcal{P}} &= (1 - w_f) \sigma_T\\
\frac{\sigma_{\mathcal{P}}}{\sigma_T} &= 1 - w_f\\
w_f &= 1 - \frac{\sigma_{\mathcal{P}}}{\sigma_T}
\end{align*}
$$
Substituting this expression for $w_f$ into the expected growth rate equation:
$$
\begin{align*}
\mathbb{E}[g_{\mathcal{P}}] &= w_f g_f + (1 - w_f) \mathbb{E}[g_T]\\
&= \left(1 - \frac{\sigma_{\mathcal{P}}}{\sigma_T}\right) g_f + \frac{\sigma_{\mathcal{P}}}{\sigma_T} \mathbb{E}[g_T]\\
&= g_f - \frac{\sigma_{\mathcal{P}}}{\sigma_T} g_f + \frac{\sigma_{\mathcal{P}}}{\sigma_T} \mathbb{E}[g_T]\\
&= g_f + \frac{\sigma_{\mathcal{P}}}{\sigma_T} \left(\mathbb{E}[g_T] - g_f\right)\\
&= g_f + \underbrace{\left(\frac{\mathbb{E}[g_T] - g_f}{\sigma_T}\right)}_{\text{Sharpe ratio T.P.}}\;\sigma_{\mathcal{P}}\quad\blacksquare
\end{align*}
$$
This is a __linear relationship__ between expected growth rate and risk. The slope of this line is the __Sharpe ratio of the tangent portfolio__, which measures the excess return per unit of risk for that optimal risky portfolio.

To find the tangent portfolio, we identify the risky portfolio that maximizes the Sharpe ratio. Geometrically, it's the point where a line from the risk-free rate is tangent to the risky-only efficient frontier. Using the single index model, we solve the optimization problem for the weights $w_i$ of the risky assets in portfolio $\mathcal{P}$ that maximizes the Sharpe ratio:
$$
\boxed{
\begin{align*}
\text{maximize} &\quad \frac{\mathbb{E}[g_{\mathcal{P}}] - g_f}{\sigma_{\mathcal{P}}} = \frac{\alpha_{\mathcal{P}} + \beta_{\mathcal{P}}\;\mathbb{E}[g_M] - g_f}{\sigma_{\mathcal{P}}}\\
\text{subject to} &\quad \sum_{i\in\mathcal{P}}w_{i} = 1\\
&\quad w_{i} \geq 0 \qquad \forall{i}\in\mathcal{P}
\end{align*}}
$$
The portfolio $\mathcal{P}$ that solves this optimization problem is the tangent portfolio $T$. Once found, its Sharpe ratio $\frac{\mathbb{E}[g_T] - g_f}{\sigma_T}$ becomes the slope of the CAL.

> __Practical Notes__
>
> __Concentration constraints:__ In practice, unconstrained tangent portfolio optimization often produces extreme weights in a few assets, especially with imprecise estimates. Practitioners commonly impose concentration constraints like $0 \leq w_i \leq u$ to force diversification. While these constraints theoretically reduce the Sharpe ratio, they often improve out-of-sample performance by making portfolios more robust to estimation error.

> __Tricky problem:__ Maximizing the Sharpe ratio turns out to be surprisingly tricky, because the objective function is a ratio of two functions that depend on the weights $w_i$ (with a quadratic in the denominator). The approach that we use to solve this problem is to transform it in a Second Order Cone Program (SOCP), which can be solved efficiently using modern optimization software, such [as the COSMO.jl package in Julia](https://github.com/oxfordcontrol/COSMO.jl.git).

__However, the Sharpe ratio is more general than just the Tangent Portfolio__. Any portfolio risky-asset $\mathcal{P}$ has its own Sharpe ratio, which is a measure of the risk-growth trade-off of that specific portfolio. Thus, the Sharpe ratio provides a standardized (dimensionless) way to compare different portfolios or investment strategies, regardless of their individual compositions.

___

## Allocation Drift
Once we have established an optimal portfolio, market fluctuations can cause the asset weights to deviate from their target (optimal) allocations over time. This phenomenon, known as __portfolio drift__, can lead to unintended risk exposures and suboptimal performance.


### Why does drift occur?
The weight (or _dollar fraction_) of asset $i$ in portfolio $\mathcal{P}$ is given by:
$$
\omega_{i} = \frac{n_{i}\cdot{S}_{i}}{\sum_{j\in\mathcal{P}}n_{j}\cdot{S}_{j}}\qquad\forall{i}\in\mathcal{P}
$$
where $n_{i}$ denotes the number of shares of asset $i$, and $S_{i}$ denotes the share price of asset $i$. The numerator is the value of asset $i$ in the portfolio, while the denominator is the portfolio’s total value. Thus, because share prices change, the optimal allocation $\omega_{i}$ drifts over time if the number of shares of each asset stays the same. 

Suppose we have a portfolio $\mathcal{P}$ with $N$ risky assets, each initially allocated $w_{i}^{(0)}$ of the budget $W^{(0)}$ at time $t=0$, where the price of each asset at the time of allocation is given by $S_{i}^{(0)}$. After some time, the prices of these assets change, leading to new prices $S_{i}^{(t)}$ at time $t$. If we do not adjust the number of shares $n_{i}$ held in each asset, the new weights $w_{i}^{(t)}$ will be:
$$
\begin{align*}
w_{i}^{(t)} &= \frac{n_{i} \cdot S_{i}^{(t)}}{\sum_{j=1}^{N} n_{j} \cdot S_{j}^{(t)}} \\
&= \frac{\left(\frac{w_{i}^{(0)} \cdot W^{(0)}}{S_{i}^{(0)}}\right) \cdot S_{i}^{(t)}}{\sum_{j=1}^{N} \left(\frac{w_{j}^{(0)} \cdot W^{(0)}}{S_{j}^{(0)}}\right) \cdot S_{j}^{(t)}}\\
&= \frac{w_{i}^{(0)} \cdot \left(\frac{S_{i}^{(t)}}{S_{i}^{(0)}}\right)}{\sum_{j=1}^{N} w_{j}^{(0)} \cdot \left(\frac{S_{j}^{(t)}}{S_{j}^{(0)}}\right)}\\
& = \frac{w_{i}^{(0)} \cdot (1 + R_{i}^{(t)})}{\sum_{j=1}^{N} w_{j}^{(0)} \cdot (1 + R_{j}^{(t)})}
\end{align*}
$$
where $R_{i}^{(t)} = ({S_{i}^{(t)} - S_{i}^{(0)}})/{S_{i}^{(0)}}$ is the __fractional return__ of asset $i$ from time $0$ to time $t$. With the course convention $r_i^{(t)}=(\Delta t)g_i^{(t)}$, the exact gross price relative is $1+R_i^{(t)}=\exp((\Delta t)g_i^{(t)})$; using $1+(\Delta t)g_i^{(t)}$ would only be a small-move approximation. The new weight $w_{i}^{(t)}$ depends on the initial weight $w_{i}^{(0)}$ and the returns (price changes) of all assets in the portfolio. 

The only way for the weights to remain unchanged ($w_{i}^{(t)} = w_{i}^{(0)}$) is if all assets experience __exactly__ the same return over the time period, i.e., $R_{i}^{(t)} = R_{j}^{(t)}$ for all $i,j \in \mathcal{P}$ (which is highly unlikely in practice).

__TL;DR__: Even if we started with an optimal allocation, the weights will naturally drift away from their targets due to differing asset returns. But why does this matter?

> __Why does this matter?__: Price changes move realized weights away from their targets. Continuously restoring the model weights would require trading after every price change, but that is generally neither necessary nor optimal once transaction costs, taxes, estimation error, liquidity, and discrete monitoring are included. Practical policies rebalance periodically or when weights cross tolerance bands.

Continuous rebalancing is impractical for many reasons, e.g., transaction costs, taxes incurred from frequent trading, data processing requirements, etc. Therefore, we need to explore strategies that allow us to rebalance our portfolio effectively without the need for constant adjustments.

Let's do an example to illustrate portfolio drift and the need for rebalancing.

> __Example__
> 
> [▶ Let's compute the drift of a risky asset maximum Sharpe portfolio.](CHEME-5660-L7b-Example-Portfolio-Drift-Fall-2026.ipynb). In this example, we'll select an optimal risky asset portfolio (computed using the single index model) and compute its drift over time. We'll analyze how the drift impacts the expected growth rates and the associated risk of the portfolio and discuss its implications for investment strategies.

___

## When Optimal Fails: Weaknesses of Mean-Variance Optimization
The minimum-variance framework is elegant, but in practice it has __three persistent weaknesses__:

* **Input sensitivity:** The optimizer treats estimated expected return and expected risk as if they were exact, so small estimation errors produce large weight swings, [overweighting assets with overstated returns and underweighting those with understated returns](https://doi.org/10.2469/faj.v45.n1.31) (Michaud's _error maximizer_); [Chopra and Ziemba](https://doi.org/10.3905/jpm.1993.409440) further showed that errors in expected returns dominate errors in variances by a wide margin.
* **Weight concentration:** Without meaningful constraints, minimum-variance portfolios often concentrate in a small number of assets. The quadratic objective $\mathbf{w}^{\top}\boldsymbol{\Sigma}_g\mathbf{w}$ strongly rewards slightly lower variance, so a portfolio that appears diversified can end up placing most of its weight in just a few names.
* **Assumption fragility:** The framework assumes returns are drawn from a stable multivariate distribution, but that assumption often fails when it matters most. During crises and regime shifts, correlations rise, volatilities jump, and expected returns change, so a portfolio that looked optimal in calm markets can perform poorly when conditions shift.

These weaknesses split cleanly into what we can __trust__ and what we must __verify__. We can trust the QP solution given well-posed inputs. However, the expected-return vector, the covariance matrix, and the concentration cap are the three levers the optimizer silently accepts as truth, and they are the three places an audit should look first. 

### Forward Stress Testing via Hybrid Monte Carlo

Classical stress testing perturbs the inputs (correlations, prices, costs) and re-solves the optimization; we instead fix the allocation and sample many *forward futures* using a _calibrated generative model_. Given an allocation $\mathbf{w}$ and initial wealth $W_{\mathcal{P}}(0) = B_0$, the share count $n_i = w_i\,W_{\mathcal{P}}(0)/S_i(t_0)$ is fixed at $t_0$ and held unchanged for the entire horizon (no rebalancing), so wealth at any future time $t_k$ is the share-weighted price aggregate:
$$\boxed{W_{\mathcal{P}}(t_k) = \sum_{i=1}^{N} n_i\,S_i(t_k), \quad\text{where}\quad n_i = \frac{w_i\,W_{\mathcal{P}}(0)}{S_i(t_0)}}$$

We draw $n_{\text{paths}}$ synthetic price trajectories $\bigl\{S_i^{(p)}(t_k)\bigr\}_{p=1}^{n_{\text{paths}}}$ from our calibrated generative model. This model has some cool features:

* __Regime-switching market path:__ We fit the generative model to 10 years of daily SPY data, modeling the market factor with 100 hidden states (ranging from bull and bear) plus jumps, producing aggregate paths with fat tails, volatility clustering and regime shifts.
* __Per-ticker idiosyncratic draws:__ each ticker contributes a demeaned generative model-simulated residual scaled to match its SIM $\sigma_{\varepsilon}$. The demeaning prevents double-counting of alpha that the SIM's $\alpha_i$ term already carries.
* __Student-$t$ copula rank-reordering:__ the per-ticker residuals are rank-reordered with a Student-$t$ copula calibrated on the training cross-section, preserving heavy tails and cross-sectional dependence that a naive independent-draws simulation would miss.

Together the three ingredients give the stress-test paths the stylized facts of real returns: fat tails, volatility clustering, and regime shifts. The terminal-wealth collection $\bigl\{W_{\mathcal{P}}^{(p)}(T)\bigr\}_{p=1}^{n_{\text{paths}}}$ is the **empirical distribution of outcomes** the rest of the scorecard works with. 

> __Model-risk note:__
>
> A generative model of this power is also a new source of model risk. An auditor or model-risk officer reviewing this pipeline will ask three things: was the generator calibrated on a regime comparable to the one we intend to deploy in, do the simulated tails match the fatness of the training data (QQ and tail-index checks), and how many paths support the reported CVaR standard error? 
> 
> The practical sample-size rule is straightforward: if the CVaR standard error exceeds about 1% of the CVaR estimate, the number is not yet stable enough to publish. Rerun with more paths until the ratio falls below the threshold.

___

## Portfolio NPV and Tail-Risk Metrics
Let's start by thinking about an economically meaningful question: _did the portfolio beat what the same dollars would have earned sitting in a risk-free or alternative asset?_ The right tool for that comparison is the **Net Present Value (NPV)** of the portfolio, defined for a constant continuous-compounding discount rate $\bar r$ over horizon $T$ as:

$$\boxed{
\text{NPV}(\bar r, T) = \underbrace{-W_{\mathcal{P}}(0)}_{\text{Investment}} + \underbrace{W_{\mathcal{P}}(T)\,e^{-\bar r\,T}}_{\text{Terminal Wealth}}
}$$

The first term is the cash outflow at $t_0$ (the initial investment); the second is the terminal portfolio wealth discounted back to today's dollars. Dividing by the initial wealth gives the **scaled NPV** (the fractional excess in present value per dollar invested):

$$\boxed{\frac{\text{NPV}(\bar r, T)}{W_{\mathcal{P}}(0)} = \frac{W_{\mathcal{P}}(T)}{W_{\mathcal{P}}(0)}\,e^{-\bar r\,T} - 1}$$

For stress-test evaluation we set $\bar r = g_f$, the **risk-free growth rate**. If $\text{NPV} > 0$ the risky portfolio **beats** a risk-free zero-coupon investment in present-value terms; $\text{NPV} < 0$ means the portfolio **underperforms** the risk-free baseline (we would have done better holding cash earning $g_f$); and $\text{NPV} = 0$ is by construction the risk-free portfolio itself (the *zero we measure against*).

The generative Monte Carlo gives us a distribution $\{W_{\mathcal{P}}^{(p)}(T)\}$ at the terminal time $t=T$, thus, we can compute the NPV for each path, which gives us a $\{\text{NPV}^{(p)}\}$ distribution. The key value the we can compute from this distribution is the **NPV-fail rate**, i.e., the fraction of paths on which the portfolio underperformed the risk-free baseline:

$$\boxed{\mathbb{P}\bigl(\text{NPV} < 0\bigr) = \mathbb{P}\Bigl(W_{\mathcal{P}}(T) < W_{\mathcal{P}}(0)\,e^{g_f\,T}\Bigr)}$$

The NPV summarizes whether the portfolio beat cash; the **left-tail metrics** summarize how bad the worst outcomes are. We track three: Value-at-Risk (VaR) for the threshold, Conditional VaR (CVaR) for the average loss beyond that threshold, and Maximum Drawdown for the worst peak-to-trough decline along the path.

> __Value-at-Risk (VaR) at level $\alpha$:__
>
> The Value-at-Risk at level $\alpha$ is the wealth threshold below which only an $\alpha$ fraction of paths end:
>
> $$\boxed{\text{VaR}_{\alpha}(W_T) = \inf\bigl\{w \in \mathbb{R} : \,\mathbb{P}\bigl(W_T \leq w\bigr) \geq \alpha\bigr\}}$$
>
> Equivalently, $\text{VaR}_{\alpha}$ is the $\alpha$-quantile of the terminal-wealth distribution. For $\alpha = 5\%$ it answers: _what is the worst case I should expect 95% of the time?_

VaR identifies the threshold but says nothing about the severity of losses beyond it; the CVaR fills that gap by averaging over the tail.

> __Conditional VaR (CVaR) / Expected Shortfall:__
>
> The Conditional VaR at level $\alpha$ is the **mean** terminal wealth conditional on being in the worst $\alpha$ tail:
>
> $$\boxed{\text{CVaR}_{\alpha}(W_T) = \mathbb{E}\bigl[\,W_T \,\big|\, W_T \leq \text{VaR}_{\alpha}(W_T)\,\bigr]}$$
>
> CVaR is also known as **Expected Shortfall (ES)** and is a *coherent* risk measure (it satisfies sub-additivity, which VaR does not), so it is the metric of choice for modern tail-risk reporting. Given $n_{\text{paths}}$ Monte Carlo samples, the tail contains $n_{\text{tail}} = \lfloor \alpha\,n_{\text{paths}}\rfloor$ paths; the plug-in estimator and its analytical standard error are given by:
>
> $$\boxed{\widehat{\text{CVaR}}_{\alpha} = \frac{1}{n_{\text{tail}}}\sum_{p\,\in\,\text{tail}} W_T^{(p)}, \qquad \text{SE}\bigl(\widehat{\text{CVaR}}_{\alpha}\bigr) \approx \frac{\mathrm{std}(\text{tail})}{\sqrt{n_{\text{tail}}}}}$$
>
> Reporting the standard error alongside the point estimate tells us whether $n_{\text{paths}}$ is large enough to trust the number; a rule of thumb is to rerun with more paths if the SE-to-CVaR ratio exceeds about 1%.

VaR and CVaR both summarize the terminal-wealth distribution, but neither captures how painful the journey was; the maximum drawdown measures the worst peak-to-trough decline along each path.

> __Maximum Drawdown:__
>
> Given a path-wise wealth series $W_{\mathcal{P}}^{(p)}(t)$, define the running peak as $\hat{W}^{(p)}(t) = \max_{s \leq t} W_{\mathcal{P}}^{(p)}(s)$. The maximum drawdown along that path is the deepest peak-to-trough fractional decline:
>
> $$\boxed{\text{MaxDD}^{(p)} = \max_{t}\;\frac{\hat{W}^{(p)}(t) - W_{\mathcal{P}}^{(p)}(t)}{\hat{W}^{(p)}(t)}}$$
>
> Drawdown is path-wise, not terminal: a portfolio can end the horizon at break-even after a 30% mid-path drawdown that would have triggered margin calls in real life. The scorecard reports both the median and the 95th percentile of $\text{MaxDD}^{(p)}$ across the $n_{\text{paths}}$ futures.

Let's look at an example that demonstrates how to compute these tail-risk metrics from the Monte Carlo distribution of terminal wealth.

> __Example__
> 
> [▶ Let's compute the net present value and tail-risk metrics for our portfolio](./CHEME-5660-L7b-Example-StressTest-MinVar-Fall-2026.ipynb). let's stress test the minimum-variance portfolio we constructed in the previous example. We use the hybrid Monte Carlo to generate a distribution of terminal wealth outcomes, and then compute the NPV distribution, the NPV-fail rate, and the tail-risk metrics (VaR, CVaR, Max Drawdown) to evaluate the portfolio's performance under stress.
>  

For more information on the hybrid single index regime-switching model and how to calibrate it, check out the [discrete generative model method paper](https://arxiv.org/abs/2603.10202) and a forthcoming paper on the multiasset hybrid Monte Carlo method (MHMC) that we use in the scorecard (preprint available from varnerlab on request).

___

## The Baseline Scorecard

Before we can evaluate any improvements (Sessions 2-4), we need a _baseline_: a quantitative record of how the classical fixed-weight minimum-variance allocation behaves across the hybrid Monte Carlo ensemble. The scorecard tracks **seven** metrics, computed across all $n_{\text{paths}}$ synthetic futures:

| Metric | What it tells us |
|:-------|:-----------------|
| Median NPV | Excess wealth over the risk-free baseline in present-value dollars |
| NPV-fail rate | Fraction of futures where the portfolio underperformed risk-free |
| Median terminal wealth | Center of the terminal-wealth distribution |
| VaR at 5% | Worst-case wealth threshold for 95% of futures |
| CVaR at 5% (with SE) | Average wealth in the worst 5% tail, with sampling uncertainty |
| Max drawdown (median, P95) | Worst peak-to-trough decline along each path |
| Median Sharpe | Risk-adjusted growth summary across paths |

Each metric was formally defined in the previous section. The table above is the format we use to compare portfolios across all four sessions. The example notebook computes these seven core metrics plus a few derived views (P95 drawdown, NPV scaled as a percentage of $B_0$, and CVaR of NPV) that are useful for inspection but redundant for the headline comparison.

Turnover and trading-cost metrics, staples of classical scorecards, are zero by construction in the portfolio-construction unit because the allocation is fixed for the entire horizon and we operate in the frictionless regime. They become meaningful in [the adaptive-allocation unit](../../week-13/L13a/CHEME-5660-L13a-Lecture-Adaptive-Portfolio-Rebalancing-Fall-2026.ipynb) when the AI rebalancing engine introduces time-varying weights and transaction costs.

The risk-free row of the scorecard is intentionally degenerate: deterministic terminal wealth $B_0\,e^{g_f T}$, zero drawdown, zero Sharpe, and $\text{NPV} = 0$. It is not a competitor; it is the zero line against which every risky portfolio is measured. A risky portfolio wins if its median NPV is positive _and_ its NPV-fail rate is acceptably low.

The baseline scorecard is computed in the [stress-test example notebook](./CHEME-5660-L7b-Example-StressTest-MinVar-Fall-2026.ipynb) and persisted to disk as the bar that the adaptive-allocation unit's adaptive rebalancing engine must clear.

___

## The Rebalancing Engine

The utility allocator gives us a principled way to compute positions at any point in time, i.e., in any market condition. The rebalancing engine wraps this allocator in a daily loop with explicit **safety rules** (trigger rules) that encode human judgment about acceptable risk into the machine's decision loop. 

The three trigger rules are the guardrails that constrain the daily loop. Each rule encodes a specific piece of human judgment about acceptable behavior:

| Rule | Parameter | What It Does |
|------|-----------|-------------|
| **Drawdown Limit** | $d_{\max}$ (e.g., 10%) | If portfolio wealth drops more than $d_{\max}$ from its peak, the engine de-risks to 100% cash. This is a circuit breaker. |
| **Turnover Cap** | $\tau_{\max}$ (e.g., 50%) | If the proposed rebalance would trade more than $\tau_{\max}$ of portfolio value, the trades are scaled down proportionally. This controls transaction costs. |
| **Reallocation Schedule** | $b_t \in \{0, 1\}$ | The binary schedule determines _which days_ the engine is allowed to rebalance. Daily, weekly, or event-driven. |

Without these rules the engine is unconstrained: it could rebalance every day (expensive), ignore a crash (catastrophic), or flip positions on noise in the $\lambda$ sentiment signal. The rules are the bridge between autonomous operation and investment-committee oversight.

In L15b, we extend these rules with human override protocols and escalation procedures for production deployment. For now, we implement this engine and evaluate it against the the portfolio-construction unit baselines.

This is a *backtest* algorithm: it assumes the complete market history (either a single realized path or one draw from a synthetic Monte Carlo ensemble) is available in advance. [L15a](../../week-15/L15a/CHEME-5660-L15a-Lecture-Autotrader-Validation-Fall-2026.ipynb) lifts this assumption to an *online* setting where data arrives one day at a time and the allocator must update on the fly.


<div>
    <center>
        <img src="figs/Fig-RebalancingEngine-Architecture.svg" width="1200" alt="Closed-loop architecture of the the adaptive-allocation unit rebalancing engine"/>
    </center>
</div>

**Figure 1:** The the adaptive-allocation unit rebalancing engine as a closed-loop OODA cycle (Observe / Orient / Decide / Act). Each bar close triggers one full traversal of the cycle. **1. Observe** reads the new bar's prices and refreshes the SIM regression and the EMA-crossover sentiment signal $\lambda_t$. **2. Orient** is the AI step: a `tanh`-bounded non-linear map turns features $(\alpha_i, \beta_i, \lambda_t)$ into per-asset preferences $\gamma_i \in (-1, 1)$. **3. Decide** computes target shares $n_i^{\star}$ from the Cobb-Douglas closed form, then checks the committee-approved guardrails (rebalance schedule $b_t$, drawdown gate $d_{\max}$, turnover cap $\tau_{\max}$). **4. Act** trades or holds, realizes transaction costs, and updates the portfolio state $W_t$. The the portfolio-construction unit minimum-variance allocation is one shot of Observe + Decide at $t=0$ followed by frozen shares forever; the the adaptive-allocation unit engine re-fires this loop on every bar close, which is what gives it the ability to respond to a regime shift the static optimizer cannot.

### Algorithm: Rebalancing Engine (Backtest)

__Initialize__: Given a **complete** market price path $\{S_t\}_{t=1}^{T}$ for a broad index, a ticker universe of $K$ assets with price matrix $\mathbf{P} \in \mathbb{R}^{T \times K}$ where $P_{t,i}$ is the close price of ticker $i$ on day $t$, SIM parameters $\{\alpha_i, \beta_i, \sigma_{\varepsilon,i}\}$, budget $B$, risk floor $\epsilon$, trigger parameters $(d_{\max}, \tau_{\max})$, and a reallocation schedule $\{b_t\}_{t=1}^{T}$ with $b_t \in \{0, 1\}$ ($b_t = 1$ marks a rebalance day; we use $b$ here so the symbol does not collide with the bandit-arm $a_t$ in [L15a](../../week-15/L15a/CHEME-5660-L15a-Lecture-Autotrader-Validation-Fall-2026.ipynb)). 

Compute the short-term EMA ($L_{\text{short}} = 21$) and long-term EMA ($L_{\text{long}} = 63$) of $S_t$; define the warmup offset $t_0 \gets L_{\text{short}} + L_{\text{long}}$. Build the sentiment series $\lambda_t \gets -G \cdot (\bar{S}^{\text{short}}_t / \bar{S}^{\text{long}}_t - 1)$ and smooth the market log-growth into $g_{m,t}$ via an EMA with window $L_{\text{growth}} = 10$ (which warms up well inside $t_0$, so it does not need to enter the offset). Assemble the context model containing $B$, the ticker universe, $\mathbf{P}$, the SIM parameters, $\epsilon$, and $g_{m,t}$. Compute the initial allocation via Cobb-Douglas utility maximization.

__Daily Loop__: For $t = t_0 + 1, \ldots, t_0 + T$ __do__:

1. Read the reallocation flag $b_t$ from the schedule.
2. If $b_t = 1$ (rebalance day), mark the portfolio to market at today's prices to set the budget $B \gets W_{\mathcal{P}}(t) = \text{cash} + \sum_i n_{i,t-1}\,P_{i,t}$ (no shares sold yet), then:
    - Check the drawdown trigger: if the running drawdown exceeds $d_{\max}$, de-risk to 100% cash and skip to the next day.
    - Otherwise, update $\lambda_t$, compute new preference weights $\gamma_{i,t}$, and solve the Cobb-Douglas allocation for the target shares $n_{i,t}^{\star}$.
    - Trade only the delta $\Delta n_i = n_{i,t}^{\star} - n_{i,t-1}$ and pay a transaction cost $c\,\sum_i |\Delta n_i|\,P_{i,t}$ debited from cash, where $c$ is a dimensionless per-trade rate (USD of cost per USD of notional traded). Our examples use $c = 5\times 10^{-4}$ (5 bps).
    - Check the turnover cap: if $\sum_i |\Delta n_i|\,P_{i,t} > \tau_{\max}\,W_{\mathcal{P}}(t)$, scale every leg by $s = \tau_{\max}\,W_{\mathcal{P}}(t) \,/\, \sum_i |\Delta n_i|\,P_{i,t} \in (0, 1)$, applying $s$ to both the share deltas $\Delta n_i$ and the cash delta so the budget invariant $\text{cash} + \sum_i n_i\,P_{i,t} = W_{\mathcal{P}}(t)$ holds. The realized turnover after scaling is exactly $\tau_{\max}\,W_{\mathcal{P}}(t)$.
3. If $b_t = 0$ (hold day), propagate the prior allocation unchanged.

__Output__: Return the history indexed by trading day: positions, preference weights, cash, and total wealth.

The output history is what every scorecard and diagnostic example in this session consumes.

> __Example__
>
> [▶ Let's wire the allocator into a rebalancing engine and produce a scorecard](../../week-13/L13a/CHEME-5660-L13a-Example-Adaptive-Rebalancing-Scorecard-Fall-2026.ipynb). We run the Cobb-Douglas engine with trigger rules on a synthetic market path, compare it head-to-head against the the portfolio-construction unit baselines, and sweep the CES elasticity parameter to see how concentration tunes engine behavior.
>
> ▶ Let's evaluate the engine across many Monte Carlo futures. We generate the full Monte Carlo ensemble, compute the tail-risk scorecard (VaR, CVaR, drawdown, NPV), pair the engine path-by-path against the the portfolio-construction unit min-var baseline, and diagnose whether the engine's edge concentrates in bull regimes, bear regimes, or across the distribution uniformly.

___

## Summary
This lecture connected static portfolio construction to forward stress testing and guarded rebalancing.

> __Key Takeaways:__
>
> * __Optimal weights are conditional outputs:__ They depend on estimated rewards, risks, constraints, and the market state used to construct them.
> * __Stress testing is part of portfolio design:__ Drawdown, tail loss, turnover, and transaction costs reveal weaknesses that an in-sample efficient frontier can hide.
> * __Rebalancing is a control problem:__ A useful engine observes portfolio state, evaluates a target, applies explicit triggers and limits, and records the resulting action.

These ideas prepare us to distinguish a mathematically optimal allocation from a portfolio process that can survive changing conditions.
___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___